In [ ]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

In [ ]:
# Drop constant columns (no variance in this dataset)
df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'], inplace=True)

# Drop duplicates
df.drop_duplicates(inplace=True)

# Encode target variable
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Encode binary categorical column
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})

# One-hot encode remaining categoricals
df = pd.get_dummies(df, columns=['BusinessTravel', 'Department', 'EducationField',
                                  'Gender', 'JobRole', 'MaritalStatus'],
                    drop_first=True)

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


In [ ]:
# Ratio features
df['PromotionToTenureRatio']  = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)
df['IncomePerYearAtCompany']  = df['MonthlyIncome'] / (df['YearsAtCompany'] + 1)

# Tenure bucket → OHE immediately
df['TenureBucket'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, float('inf')],
    labels=['0-2', '3-5', '6-10', '10+']
)
df = pd.get_dummies(df, columns=['TenureBucket'], drop_first=True)
tenure_dummies = [c for c in df.columns if c.startswith('TenureBucket_')]
for col in tenure_dummies:
    df[col] = df[col].astype(int)

# Overtime flag (explicit alias of the already-binary OverTime column)
df['OvertimeFlag'] = df['OverTime']

# Interaction features
df['OverTime_x_JobSatisfaction']          = df['OverTime'] * df['JobSatisfaction']
df['DistanceFromHome_x_WorkLifeBalance']  = df['DistanceFromHome'] * df['WorkLifeBalance']

new_cols = ['PromotionToTenureRatio', 'IncomePerYearAtCompany',
            'OvertimeFlag', 'OverTime_x_JobSatisfaction',
            'DistanceFromHome_x_WorkLifeBalance'] + tenure_dummies
print(f"Shape after feature engineering: {df.shape}")
df[new_cols].head()


In [ ]:
ordinal_maps = {
    'Education':              {1: 'Below College', 2: 'College', 3: 'Bachelor', 4: 'Master', 5: 'Doctor'},
    'EnvironmentSatisfaction':{1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobInvolvement':         {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'JobSatisfaction':        {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'PerformanceRating':      {1: 'Low', 2: 'Good', 3: 'Excellent', 4: 'Outstanding'},
    'RelationshipSatisfaction':{1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'},
    'WorkLifeBalance':        {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'},
}

ordinal_orders = {
    'Education':              ['Below College', 'College', 'Bachelor', 'Master', 'Doctor'],
    'EnvironmentSatisfaction':['Low', 'Medium', 'High', 'Very High'],
    'JobInvolvement':         ['Low', 'Medium', 'High', 'Very High'],
    'JobSatisfaction':        ['Low', 'Medium', 'High', 'Very High'],
    'PerformanceRating':      ['Low', 'Good', 'Excellent', 'Outstanding'],
    'RelationshipSatisfaction':['Low', 'Medium', 'High', 'Very High'],
    'WorkLifeBalance':        ['Bad', 'Good', 'Better', 'Best'],
}

# Map integers → ordered Categorical strings
for col, mapping in ordinal_maps.items():
    df[col] = pd.Categorical(
        df[col].astype(int).map(mapping),
        categories=ordinal_orders[col],
        ordered=True
    )

# OHE the ordinal columns
df = pd.get_dummies(df, columns=list(ordinal_maps.keys()), drop_first=True)

# Convert all boolean columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()


In [ ]:
import pickle

with open('df_cleaned.pkl', 'wb') as f:
    pickle.dump({'df': df}, f)
print(f"Saved df_cleaned.pkl — shape: {df.shape}")